# DataScienceJobs — Exploration Notebook

Use this notebook to explore the live database, spot trends, and diagnose data quality issues.
This notebook reflects the **3-Stage Medallion Architecture** implemented in the `pipeline/` directory.

## Medallion Data Layers

| Layer | Columns | Meaning |
|---|---|---|
| 🥉 **Bronze** | `*_raw` | Original API values, captured once at ingestion, never modified. |
| 🥈 **Silver** | Core Columns | Cleaned, normalized, and enriched values used by the dashboard. |
| 🥇 **Gold** | `v_jobs_enriched` | Aggregated views joining job postings with company metadata. |

### Column Provenance Details

In [ ]:
import pandas as pd

PROVENANCE = [
    # column                 layer      transformation                                        source class
    ("job_id",               "🥉 Bronze", "Source prefix (adzuna_, etc.) to prevent collisions", "BaseJobSource"),
    ("*_raw",                "🥉 Bronze", "Zero processing; direct 1:1 capture from API response", "BaseJobSource"),
    
    ("job_description",      "🥈 Silver", "HTML stripped, collapsed whitespace",                "JobTransformer"),
    ("location_city",        "🥈 Silver", "Accent-stripped, alias-resolved, noise removed",     "JobTransformer"),
    ("location_state",       "🥈 Silver", "Normalised + inferred from city or description",     "JobTransformer"),
    ("location_country",     "🥈 Silver", "CA -> Canada; null -> Anywhere",                       "JobTransformer"),
    ("salary_min/max",       "🥈 Silver", "Hourly rates converted to annual (x2080)",           "JobTransformer"),
    ("is_remote",            "🥈 Silver", "Inferred from description if API flag is missing",  "JobTransformer"),
    
    ("skills_tags",          "🥈 Silver", "Keyword match against 73-skill list",            "JobTransformer"),
    ("years_experience_min", "🥈 Silver", "Regex extraction from description",                 "JobTransformer"),
    ("seniority",            "🥈 Silver", "Title keyword match -> years brackets -> Mid",     "JobTransformer"),

    ("id",                   "🗄️ DB",     "UUID primary key",                                   "PostgreSQL"),
    ("fetched_at",           "🗄️ DB",     "Timestamp of pipeline run",                         "PostgreSQL"),
]

prov_df = pd.DataFrame(PROVENANCE, columns=["column", "layer", "transformation", "source module"])

COLOR_MAP = {
    "🥉 Bronze": "background-color: #1a3a52; color: #90caf9",
    "🥈 Silver": "background-color: #2d3b1a; color: #a5d6a7",
    "🗄️ DB":     "background-color: #2a1a3b; color: #ce93d8",
}

def _style_row(row):
    style = COLOR_MAP.get(row["layer"], "")
    return [style] * len(row)

prov_df.style.apply(_style_row, axis=1).set_properties(**{"font-size": "12px"}).hide(axis="index")

In [ ]:
import sys, os
from pathlib import Path

# ── Set working directory to project root ───────────────────────────────────
# This ensures pipeline.* imports and .env loading work consistently
root_dir = Path(os.getcwd()).parent
os.chdir(root_dir)
sys.path.append(str(root_dir))

import pandas as pd
from pipeline.config import SUPABASE_URL, SUPABASE_SERVICE_KEY
from supabase import create_client

client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

# ── Load job postings (including the 16 new Bronze columns) ──────────────────
resp = client.table("job_postings").select("*").execute()
df = pd.DataFrame(resp.data)

df["posted_at"] = pd.to_datetime(df["posted_at"], utc=True, errors="coerce")
df["fetched_at"] = pd.to_datetime(df["fetched_at"], utc=True, errors="coerce")

def _source(jid):
    for p in ("adzuna_", "theirstack_", "serpapi_"): 
        if jid.startswith(p): return p.rstrip("_")
    return "jsearch"
df["source"] = df["job_id"].apply(_source)

print(f"Loaded {len(df):,} jobs with {df.columns.size} columns (Bronze + Silver)")

## Data Quality: Bronze vs Silver
Compare the raw API values against our cleaned/transformed versions.

In [ ]:
cols = ["title", "title_raw", "location_city", "location_city_raw", "salary_min", "salary_min_raw", "is_remote", "is_remote_raw"]
df[cols].sample(10)

In [ ]:
# Jobs per source
df["source"].value_counts()

In [ ]:
# Seniority distribution
df["seniority"].value_counts(dropna=False)

In [ ]:
# Top 20 Skills (Silver layer enrichment check)
from collections import Counter
skill_counts = Counter(sk for tags in df["skills_tags"].dropna() for sk in tags)
pd.Series(skill_counts).sort_values(ascending=False).head(20)